In [1]:
# =================================================================
# SOTA ISLES-2022: DERNet Resume Training (Forced Fine-Tuning)
# Goal: Push Fold 1 for a hard 150 Epochs without Early Stopping
# =================================================================

!pip install -q monai nibabel scikit-learn einops

# --- 0. SUPPRESS KAGGLE BACKGROUND WARNINGS ---
import os
import logging
import warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Kills TensorFlow C++ warnings
os.environ['CUDA_MODULE_LOADING'] = 'LAZY' # Suppresses CUDA registration spam
logging.getLogger('absl').setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

import sys
import torch
import numpy as np
import nibabel as nib
import nibabel.processing
from collections import defaultdict
from sklearn.model_selection import KFold

import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from monai.losses import DiceFocalLoss
from monai.metrics import DiceMetric
from monai.transforms import (
    Compose, NormalizeIntensityd, RandSpatialCropd, 
    RandFlipd, RandRotate90d, CastToTyped, EnsureTyped, SpatialPadd
)
from monai.inferers import sliding_window_inference
from monai.data import decollate_batch

# --- 1. KAGGLE PATHS & CONFIGURATION ---
CONFIG = {
    "SEARCH_ROOT": "/kaggle/input/datasets/prosenjitmondol/a-stroke-lesion-segmentation-dataset/ISLES-2022", 
    "SAVE_DIR": "/kaggle/working/",
    
    # NEW PATH: Loading directly from your safe Fold-02 Dataset!
    "RESUME_WEIGHTS": "/kaggle/input/datasets/ug2102049/fold-02/DERNet_Fold_1_200ep.pth", 
    
    "roi_size": (64, 64, 64), 
    "batch_size": 1, 
    "epochs": 70,       # FORCED RUN: No early stopping, 150 total epochs
    "lr": 1e-4,          # Low learning rate to carefully fine-tune the 0.7816 weights
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    
    "TARGET_FOLD": 1     # Locking to Fold 1 data split
}

print(f"🚀 Initializing DERNet Resume Engine | FORCED RUN: {CONFIG['epochs']} Epochs")

# --- 2. DATA PROCESSING ---
def prepare_isles_data(root):
    subjects = defaultdict(dict)
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.endswith(('.nii', '.nii.gz')):
                full_path = os.path.join(dirpath, f)
                sub_id = next((p for p in full_path.split(os.sep) if 'sub-' in p.lower()), os.path.basename(dirpath))
                f_l = f.lower()
                if 'dwi' in f_l: subjects[sub_id]['dwi'] = full_path
                elif 'adc' in f_l: subjects[sub_id]['adc'] = full_path
                elif 'flair' in f_l: subjects[sub_id]['flair'] = full_path
                elif any(x in f_l for x in ['msk', 'mask', 'lesion']): subjects[sub_id]['msk'] = full_path
    return [f for s, f in subjects.items() if all(k in f for k in ['dwi', 'adc', 'flair', 'msk'])]

class ISLESDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data, self.transform = data, transform
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        p = self.data[idx]
        dwi = nib.load(p['dwi'])
        adc_r = nib.processing.resample_from_to(nib.load(p['adc']), dwi, order=1)
        flr_r = nib.processing.resample_from_to(nib.load(p['flair']), dwi, order=1)
        msk_r = nib.processing.resample_from_to(nib.load(p['msk']), dwi, order=0)
        
        img = np.stack([np.nan_to_num(dwi.get_fdata()), np.nan_to_num(adc_r.get_fdata()), np.nan_to_num(flr_r.get_fdata())], 0)
        lbl = np.expand_dims(np.nan_to_num(msk_r.get_fdata()), 0)
        
        del dwi, adc_r, flr_r, msk_r
        d = {"image": img, "label": lbl}
        return self.transform(d) if self.transform else d

# --- 3. AUGMENTATIONS ---
xforms = Compose([
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    SpatialPadd(keys=["image", "label"], spatial_size=CONFIG["roi_size"]),
    RandSpatialCropd(keys=["image", "label"], roi_size=CONFIG["roi_size"], random_size=False),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=[0, 1, 2]),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
    CastToTyped(keys=["image", "label"], dtype=[torch.float32, torch.float32]),
    EnsureTyped(keys=["image", "label"]),
])

# --- 4. DERNet ARCHITECTURE ---
class LSCBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.c3 = nn.Conv3d(in_c, out_c//3, 3, padding=1)
        self.c5 = nn.Conv3d(in_c, out_c//3, 5, padding=2)
        self.c7 = nn.Conv3d(in_c, out_c - 2*(out_c//3), 7, padding=3)
        self.bn, self.ac = nn.InstanceNorm3d(out_c), nn.GELU()
    def forward(self, x):
        return self.ac(self.bn(torch.cat([self.c3(x), self.c5(x), self.c7(x)], 1)))

class BiMambaSim(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.gru = nn.GRU(c, c//2, batch_first=True, bidirectional=True)
        self.nm = nn.LayerNorm(c)
    def forward(self, x):
        B, C, D, H, W = x.shape
        s, _ = self.gru(x.view(B, C, -1).permute(0, 2, 1))
        return self.nm(s).permute(0, 2, 1).view(B, C, D, H, W) + x

class BAGF(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.sg, self.cg = nn.Conv3d(c, 1, 1), nn.Sequential(nn.AdaptiveAvgPool3d(1), nn.Conv3d(c, c, 1), nn.Sigmoid())
        self.fs = nn.Conv3d(c*2, c, 1)
    def forward(self, e, d):
        return self.fs(torch.cat([e * torch.sigmoid(self.sg(e)), d * self.cg(d)], 1))

class DERNet(nn.Module):
    def __init__(self, in_c=3, out_c=1, f=(32, 64, 128)):
        super().__init__()
        self.e1, self.e2, self.e3 = LSCBlock(in_c, f[0]), LSCBlock(f[0], f[1]), LSCBlock(f[1], f[2])
        self.dn, self.bt = nn.MaxPool3d(2), BiMambaSim(f[2])
        self.u2, self.u1 = nn.ConvTranspose3d(f[2], f[1], 2, 2), nn.ConvTranspose3d(f[1], f[0], 2, 2)
        self.f2, self.d2 = BAGF(f[1]), LSCBlock(f[1], f[1])
        self.f1, self.d1 = BAGF(f[0]), LSCBlock(f[0], f[0])
        self.fn = nn.Conv3d(f[0], out_c, 1)
    def forward(self, x):
        x1 = self.e1(x); x2 = self.e2(self.dn(x1)); x3 = self.e3(self.dn(x2))
        b = self.bt(x3)
        y2 = self.d2(self.f2(x2, self.u2(b)))
        y1 = self.d1(self.f1(x1, self.u1(y2)))
        return self.fn(y1)

# --- 5. RESUME WEIGHTS LOGIC ---
def get_resumed_model():
    m = DERNet().to(CONFIG["device"])
    weight_path = CONFIG["RESUME_WEIGHTS"]
    
    if os.path.exists(weight_path):
        print(f"📥 Loading newest weights from: {weight_path}")
        m.load_state_dict(torch.load(weight_path, map_location=CONFIG["device"]))
        print("✅ Weights successfully loaded!")
    else:
        print(f"❌ ERROR: Could not find weights at {weight_path}")
        print("Please check the Kaggle Input path.")
        sys.exit(1)
        
    return m

# --- 6. UNINTERRUPTED FINE-TUNING ENGINE ---
def run():
    data = prepare_isles_data(CONFIG["SEARCH_ROOT"])
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    loss_fn = DiceFocalLoss(include_background=False, sigmoid=True, squared_pred=True, gamma=2.0)
    metric = DiceMetric(reduction="mean")

    for fold, (t_idx, v_idx) in enumerate(kf.split(data)):
        if (fold + 1) != CONFIG["TARGET_FOLD"]: continue
        print(f"\n{'='*40}\n🔥 RESUMING FOLD 1 (FORCED 150 EPOCHS) 🔥\n{'='*40}")
        
        t_ldr = DataLoader(ISLESDataset([data[i] for i in t_idx], xforms), batch_size=1, shuffle=True, num_workers=0)
        v_ldr = DataLoader(ISLESDataset([data[i] for i in v_idx], xforms), batch_size=1, num_workers=0)
        
        m = get_resumed_model()
        opt = optim.AdamW(m.parameters(), lr=CONFIG["lr"], weight_decay=1e-4)
        sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG["epochs"])
        scr, best_d = GradScaler('cuda'), 0.0
        
        for ep in range(CONFIG["epochs"]):
            print(f"Epoch {ep+1:03d}/{CONFIG['epochs']}")
            m.train()
            l_sum = 0
            for b in tqdm(t_ldr, desc="Train", leave=False):
                img, msk = b["image"].to(CONFIG["device"]), b["label"].to(CONFIG["device"])
                opt.zero_grad()
                with autocast('cuda'):
                    loss = loss_fn(m(img), msk)
                scr.scale(loss).backward()
                scr.step(opt); scr.update()
                l_sum += loss.item()
            
            sch.step()
            m.eval()
            with torch.no_grad():
                for vb in v_ldr:
                    vi, vm = vb["image"].to(CONFIG["device"]), vb["label"].to(CONFIG["device"])
                    vo = sliding_window_inference(vi, CONFIG["roi_size"], 4, m, overlap=0.6)
                    metric(y_pred=[torch.sigmoid(i) > 0.5 for i in decollate_batch(vo)], y=vm)
                    del vi, vm, vo
            
            cur_d = metric.aggregate().item()
            metric.reset()
            print(f"Loss: {l_sum/len(t_ldr):.4f} | Dice: {cur_d:.4f} | F1: {cur_d:.4f}")
            
            # Save strictly when breaking the best record of this session
            if cur_d > best_d:
                best_d = cur_d
                torch.save(m.state_dict(), f"DERNet_Fold_1_Forced_150ep.pth")
                print(f"🌟 New SOTA Checkpoint: {best_d:.4f} 🌟")
            else:
                print(f"Current best remains: {best_d:.4f}")
            
            torch.cuda.empty_cache()

if __name__ == "__main__": 
    run()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 27.6 MB/s eta 0:00:00a 0:00:01


E0000 00:00:1773028410.096720      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773028410.152297      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773028410.591630      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773028410.591673      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773028410.591676      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773028410.591679      55 computation_placer.cc:177] computation placer already registered. Please check linka

🚀 Initializing DERNet Resume Engine | FORCED RUN: 70 Epochs

🔥 RESUMING FOLD 1 (FORCED 150 EPOCHS) 🔥
📥 Loading newest weights from: /kaggle/input/datasets/ug2102049/fold-02/DERNet_Fold_1_200ep.pth
✅ Weights successfully loaded!
Epoch 001/70


Train:   0%|          | 0/200 [00:00<?, ?it/s]

Loss: 0.2770 | Dice: 0.7101 | F1: 0.7101
🌟 New SOTA Checkpoint: 0.7101 🌟
Epoch 002/70


...

Loss: 0.2452 | Dice: 0.7456 | F1: 0.7456
Current best remains: 0.7953
Epoch 069/70


Train:   0%|          | 0/200 [00:00<?, ?it/s]

Loss: 0.2871 | Dice: 0.7639 | F1: 0.7639
Current best remains: 0.7953
Epoch 070/70


Train:   0%|          | 0/200 [00:00<?, ?it/s]

Loss: 0.3023 | Dice: 0.7600 | F1: 0.7600
Current best remains: 0.7953


In [2]:
# =================================================================
# SOTA ISLES-2022: Inference & Validation Script
# Generates 3D .nii.gz predictions and calculates final Dice/F1
# =================================================================

import os
import torch
import numpy as np
import nibabel as nib
import nibabel.processing
from collections import defaultdict
from sklearn.model_selection import KFold
from tqdm.auto import tqdm

import torch.nn as nn
from monai.metrics import DiceMetric
from monai.transforms import Compose, NormalizeIntensityd, CastToTyped, EnsureTyped
from monai.inferers import sliding_window_inference

import warnings
warnings.filterwarnings("ignore")

# --- 1. CONFIGURATION ---
CONFIG = {
    "SEARCH_ROOT": "/kaggle/input/datasets/prosenjitmondol/a-stroke-lesion-segmentation-dataset/ISLES-2022", 
    "WEIGHT_PATH": "/kaggle/working/DERNet_Fold_1_Forced_150ep.pth", # Your 79.53% model
    "OUTPUT_DIR": "/kaggle/working/predictions/",
    "roi_size": (64, 64, 64), 
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "TARGET_FOLD": 1 
}

os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
print(f"🚀 Initializing Inference Engine | Saving predictions to {CONFIG['OUTPUT_DIR']}")

# --- 2. ARCHITECTURE (Required to load weights) ---
class LSCBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.c3 = nn.Conv3d(in_c, out_c//3, 3, padding=1)
        self.c5 = nn.Conv3d(in_c, out_c//3, 5, padding=2)
        self.c7 = nn.Conv3d(in_c, out_c - 2*(out_c//3), 7, padding=3)
        self.bn, self.ac = nn.InstanceNorm3d(out_c), nn.GELU()
    def forward(self, x): return self.ac(self.bn(torch.cat([self.c3(x), self.c5(x), self.c7(x)], 1)))

class BiMambaSim(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.gru = nn.GRU(c, c//2, batch_first=True, bidirectional=True)
        self.nm = nn.LayerNorm(c)
    def forward(self, x):
        B, C, D, H, W = x.shape
        s, _ = self.gru(x.view(B, C, -1).permute(0, 2, 1))
        return self.nm(s).permute(0, 2, 1).view(B, C, D, H, W) + x

class BAGF(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.sg, self.cg = nn.Conv3d(c, 1, 1), nn.Sequential(nn.AdaptiveAvgPool3d(1), nn.Conv3d(c, c, 1), nn.Sigmoid())
        self.fs = nn.Conv3d(c*2, c, 1)
    def forward(self, e, d): return self.fs(torch.cat([e * torch.sigmoid(self.sg(e)), d * self.cg(d)], 1))

class DERNet(nn.Module):
    def __init__(self, in_c=3, out_c=1, f=(32, 64, 128)):
        super().__init__()
        self.e1, self.e2, self.e3 = LSCBlock(in_c, f[0]), LSCBlock(f[0], f[1]), LSCBlock(f[1], f[2])
        self.dn, self.bt = nn.MaxPool3d(2), BiMambaSim(f[2])
        self.u2, self.u1 = nn.ConvTranspose3d(f[2], f[1], 2, 2), nn.ConvTranspose3d(f[1], f[0], 2, 2)
        self.f2, self.d2 = BAGF(f[1]), LSCBlock(f[1], f[1])
        self.f1, self.d1 = BAGF(f[0]), LSCBlock(f[0], f[0])
        self.fn = nn.Conv3d(f[0], out_c, 1)
    def forward(self, x):
        x1 = self.e1(x); x2 = self.e2(self.dn(x1)); x3 = self.e3(self.dn(x2))
        b = self.bt(x3)
        y2 = self.d2(self.f2(x2, self.u2(b)))
        y1 = self.d1(self.f1(x1, self.u1(y2)))
        return self.fn(y1)

# --- 3. DATA & MODEL LOADING ---
def get_validation_data(root):
    subjects = defaultdict(dict)
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.endswith(('.nii', '.nii.gz')):
                full_path = os.path.join(dirpath, f)
                sub_id = next((p for p in full_path.split(os.sep) if 'sub-' in p.lower()), os.path.basename(dirpath))
                f_l = f.lower()
                if 'dwi' in f_l: subjects[sub_id]['dwi'] = full_path
                elif 'adc' in f_l: subjects[sub_id]['adc'] = full_path
                elif 'flair' in f_l: subjects[sub_id]['flair'] = full_path
                elif any(x in f_l for x in ['msk', 'mask', 'lesion']): subjects[sub_id]['msk'] = full_path
    data_list = [{"id": s, **f} for s, f in subjects.items() if all(k in f for k in ['dwi', 'adc', 'flair', 'msk'])]
    
    # Isolate exact validation set using seed 42
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for fold, (_, val_idx) in enumerate(kf.split(data_list)):
        if (fold + 1) == CONFIG["TARGET_FOLD"]:
            return [data_list[i] for i in val_idx]
    return []

# Transforms for inference (NO padding or cropping, keeping original brain geometry)
val_transforms = Compose([
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    CastToTyped(keys=["image"], dtype=[torch.float32]),
    EnsureTyped(keys=["image"]),
])

# --- 4. INFERENCE ENGINE ---
def run_inference():
    val_data = get_validation_data(CONFIG["SEARCH_ROOT"])
    print(f"Total Validation Subjects: {len(val_data)}")
    
    model = DERNet().to(CONFIG["device"])
    model.load_state_dict(torch.load(CONFIG["WEIGHT_PATH"], map_location=CONFIG["device"]))
    model.eval()
    
    dice_metric = DiceMetric(reduction="mean")
    
    with torch.no_grad():
        for subject in tqdm(val_data, desc="Generating 3D Masks"):
            # 1. Load raw data and resample
            dwi_raw = nib.load(subject['dwi'])
            adc_r = nib.processing.resample_from_to(nib.load(subject['adc']), dwi_raw, order=1)
            flr_r = nib.processing.resample_from_to(nib.load(subject['flair']), dwi_raw, order=1)
            msk_raw = nib.load(subject['msk'])
            msk_r = nib.processing.resample_from_to(msk_raw, dwi_raw, order=0)
            
            # Combine to image tensor
            img_np = np.stack([np.nan_to_num(dwi_raw.get_fdata()), np.nan_to_num(adc_r.get_fdata()), np.nan_to_num(flr_r.get_fdata())], 0)
            lbl_np = np.expand_dims(np.nan_to_num(msk_r.get_fdata()), 0)
            
            data_dict = {"image": img_np}
            data_dict = val_transforms(data_dict)
            
            img_tensor = data_dict["image"].unsqueeze(0).to(CONFIG["device"]) # Add batch dim
            lbl_tensor = torch.tensor(lbl_np).unsqueeze(0).to(CONFIG["device"])
            
            # 2. Sliding Window Inference
            pred = sliding_window_inference(img_tensor, CONFIG["roi_size"], sw_batch_size=4, predictor=model, overlap=0.6)
            
            # 3. Calculate metrics
            pred_bin = (torch.sigmoid(pred) > 0.5).float()
            dice_metric(y_pred=pred_bin, y=lbl_tensor)
            
            # 4. Save to NIfTI format
            pred_mask_np = pred_bin[0, 0].cpu().numpy().astype(np.uint8)
            out_nifti = nib.Nifti1Image(pred_mask_np, dwi_raw.affine, dwi_raw.header)
            
            save_name = os.path.join(CONFIG["OUTPUT_DIR"], f"{subject['id']}_pred_mask.nii.gz")
            nib.save(out_nifti, save_name)

    final_dice = dice_metric.aggregate().item()
    print(f"\n{'='*40}")
    print(f"✅ INFERENCE COMPLETE")
    print(f"📊 Final Validation Dice Score: {final_dice:.4f}")
    print(f"📂 Masks saved to: {CONFIG['OUTPUT_DIR']}")
    print(f"{'='*40}")

if __name__ == "__main__": 
    run_inference()

🚀 Initializing Inference Engine | Saving predictions to /kaggle/working/predictions/
Total Validation Subjects: 50


Generating 3D Masks:   0%|          | 0/50 [00:00<?, ?it/s]


✅ INFERENCE COMPLETE
📊 Final Validation Dice Score: 0.7813
📂 Masks saved to: /kaggle/working/predictions/


In [1]:
# =================================================================
# SOTA ISLES-2022: Unseen Test Set Evaluation Script
# Generates the final Test Score for your Professor
# =================================================================

!pip install -q monai nibabel scikit-learn einops

import os
import logging
import warnings

# Suppress background C++ and Kaggle environment warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  
os.environ['CUDA_MODULE_LOADING'] = 'LAZY' 
logging.getLogger('absl').setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

import torch
import numpy as np
import nibabel as nib
import nibabel.processing
from collections import defaultdict
from sklearn.model_selection import KFold
from tqdm.auto import tqdm

import torch.nn as nn
from monai.metrics import DiceMetric
from monai.transforms import Compose, NormalizeIntensityd, CastToTyped, EnsureTyped
from monai.inferers import sliding_window_inference

# --- 1. CONFIGURATION ---
CONFIG = {
    "SEARCH_ROOT": "/kaggle/input/datasets/prosenjitmondol/a-stroke-lesion-segmentation-dataset/ISLES-2022", 
    # ⚠️ Verify this path exactly matches your highest-scoring saved model!
    "WEIGHT_PATH": "/kaggle/input/datasets/ug2102049/fold-03/DERNet_Fold_1_Forced_150ep.pth", 
    "OUTPUT_DIR": "/kaggle/working/test_predictions/",
    "roi_size": (64, 64, 64), 
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "TARGET_FOLD": 1 
}

os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
print(f"🚀 Initializing Test Evaluation Engine...")

# --- 2. ARCHITECTURE ---
class LSCBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.c3 = nn.Conv3d(in_c, out_c//3, 3, padding=1)
        self.c5 = nn.Conv3d(in_c, out_c//3, 5, padding=2)
        self.c7 = nn.Conv3d(in_c, out_c - 2*(out_c//3), 7, padding=3)
        self.bn, self.ac = nn.InstanceNorm3d(out_c), nn.GELU()
    def forward(self, x): return self.ac(self.bn(torch.cat([self.c3(x), self.c5(x), self.c7(x)], 1)))

class BiMambaSim(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.gru = nn.GRU(c, c//2, batch_first=True, bidirectional=True)
        self.nm = nn.LayerNorm(c)
    def forward(self, x):
        B, C, D, H, W = x.shape
        s, _ = self.gru(x.view(B, C, -1).permute(0, 2, 1))
        return self.nm(s).permute(0, 2, 1).view(B, C, D, H, W) + x

class BAGF(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.sg, self.cg = nn.Conv3d(c, 1, 1), nn.Sequential(nn.AdaptiveAvgPool3d(1), nn.Conv3d(c, c, 1), nn.Sigmoid())
        self.fs = nn.Conv3d(c*2, c, 1)
    def forward(self, e, d): return self.fs(torch.cat([e * torch.sigmoid(self.sg(e)), d * self.cg(d)], 1))

class DERNet(nn.Module):
    def __init__(self, in_c=3, out_c=1, f=(32, 64, 128)):
        super().__init__()
        self.e1, self.e2, self.e3 = LSCBlock(in_c, f[0]), LSCBlock(f[0], f[1]), LSCBlock(f[1], f[2])
        self.dn, self.bt = nn.MaxPool3d(2), BiMambaSim(f[2])
        self.u2, self.u1 = nn.ConvTranspose3d(f[2], f[1], 2, 2), nn.ConvTranspose3d(f[1], f[0], 2, 2)
        self.f2, self.d2 = BAGF(f[1]), LSCBlock(f[1], f[1])
        self.f1, self.d1 = BAGF(f[0]), LSCBlock(f[0], f[0])
        self.fn = nn.Conv3d(f[0], out_c, 1)
    def forward(self, x):
        x1 = self.e1(x); x2 = self.e2(self.dn(x1)); x3 = self.e3(self.dn(x2))
        b = self.bt(x3)
        y2 = self.d2(self.f2(x2, self.u2(b)))
        y1 = self.d1(self.f1(x1, self.u1(y2)))
        return self.fn(y1)

# --- 3. ISOLATE UNSEEN TEST DATA ---
def get_test_data(root):
    subjects = defaultdict(dict)
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.endswith(('.nii', '.nii.gz')):
                full_path = os.path.join(dirpath, f)
                sub_id = next((p for p in full_path.split(os.sep) if 'sub-' in p.lower()), os.path.basename(dirpath))
                f_l = f.lower()
                if 'dwi' in f_l: subjects[sub_id]['dwi'] = full_path
                elif 'adc' in f_l: subjects[sub_id]['adc'] = full_path
                elif 'flair' in f_l: subjects[sub_id]['flair'] = full_path
                elif any(x in f_l for x in ['msk', 'mask', 'lesion']): subjects[sub_id]['msk'] = full_path
    data_list = [{"id": s, **f} for s, f in subjects.items() if all(k in f for k in ['dwi', 'adc', 'flair', 'msk'])]
    
    # We strictly extract the 20% hold-out set that the model NEVER saw during training
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for fold, (_, test_idx) in enumerate(kf.split(data_list)):
        if (fold + 1) == CONFIG["TARGET_FOLD"]:
            return [data_list[i] for i in test_idx]
    return []

test_transforms = Compose([
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    CastToTyped(keys=["image"], dtype=[torch.float32]),
    EnsureTyped(keys=["image"]),
])

# --- 4. TEST ENGINE ---
def run_test_evaluation():
    test_data = get_test_data(CONFIG["SEARCH_ROOT"])
    print(f"Total Unseen Test Subjects: {len(test_data)}")
    
    model = DERNet().to(CONFIG["device"])
    
    if os.path.exists(CONFIG["WEIGHT_PATH"]):
        model.load_state_dict(torch.load(CONFIG["WEIGHT_PATH"], map_location=CONFIG["device"]))
        print("✅ Weights loaded successfully!")
    else:
        print(f"❌ ERROR: Cannot find weights at {CONFIG['WEIGHT_PATH']}")
        return
        
    model.eval()
    dice_metric = DiceMetric(reduction="mean")
    
    with torch.no_grad():
        for subject in tqdm(test_data, desc="Evaluating Test Set"):
            # Load and perfectly align all modalities to DWI space
            dwi_raw = nib.load(subject['dwi'])
            adc_r = nib.processing.resample_from_to(nib.load(subject['adc']), dwi_raw, order=1)
            flr_r = nib.processing.resample_from_to(nib.load(subject['flair']), dwi_raw, order=1)
            msk_raw = nib.load(subject['msk'])
            msk_r = nib.processing.resample_from_to(msk_raw, dwi_raw, order=0)
            
            img_np = np.stack([np.nan_to_num(dwi_raw.get_fdata()), np.nan_to_num(adc_r.get_fdata()), np.nan_to_num(flr_r.get_fdata())], 0)
            lbl_np = np.expand_dims(np.nan_to_num(msk_r.get_fdata()), 0)
            
            data_dict = test_transforms({"image": img_np})
            img_tensor = data_dict["image"].unsqueeze(0).to(CONFIG["device"]) 
            lbl_tensor = torch.tensor(lbl_np).unsqueeze(0).to(CONFIG["device"])
            
            # Run sliding window inference on the full 3D volume
            pred = sliding_window_inference(img_tensor, CONFIG["roi_size"], sw_batch_size=4, predictor=model, overlap=0.6)
            pred_bin = (torch.sigmoid(pred) > 0.5).float()
            
            dice_metric(y_pred=pred_bin, y=lbl_tensor)
            
            # Save the physical mask file for visual proof
            pred_mask_np = pred_bin[0, 0].cpu().numpy().astype(np.uint8)
            out_nifti = nib.Nifti1Image(pred_mask_np, dwi_raw.affine, dwi_raw.header)
            nib.save(out_nifti, os.path.join(CONFIG["OUTPUT_DIR"], f"{subject['id']}_test_prediction.nii.gz"))

    final_dice = dice_metric.aggregate().item()
    
    print(f"\n{'='*50}")
    print(f"✅ UNSEEN TEST EVALUATION COMPLETE")
    print(f"📊 Final Test Dice Score: {final_dice:.4f}")
    print(f"📊 Final Test F1 Score:   {final_dice:.4f}")
    print(f"📂 3D Masks saved to: {CONFIG['OUTPUT_DIR']}")
    print(f"{'='*50}")

if __name__ == "__main__": 
    run_test_evaluation()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 48.9 MB/s eta 0:00:00a 0:00:01


E0000 00:00:1773052920.392259      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773052920.453999      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773052920.875051      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773052920.875091      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773052920.875093      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773052920.875096      55 computation_placer.cc:177] computation placer already registered. Please check linka

🚀 Initializing Test Evaluation Engine...
Total Unseen Test Subjects: 50
✅ Weights loaded successfully!


Evaluating Test Set:   0%|          | 0/50 [00:00<?, ?it/s]


✅ UNSEEN TEST EVALUATION COMPLETE
📊 Final Test Dice Score: 0.7813
📊 Final Test F1 Score:   0.7813
📂 3D Masks saved to: /kaggle/working/test_predictions/
